In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import os
import re
import sys
import shutil
import zipfile
import requests
from urllib.parse import urlparse, parse_qs
from tqdm import tqdm
from urllib.parse import urlparse, parse_qs
from tqdm.notebook import tqdm
sys.path.append(r'E:\repository\dataset_tools\isds_tool\PS_data')
from vis import select_defect, esresult2yolo, esimage_merge
from img_preprocess import select_img
from deduplication_demo import filter_deduplication

In [ ]:
root_dir = r"E:\data\202502_signboard\data_annotation\ps_data\task0926"
RESULT_ALL = "http://ec2-54-46-0-164.ap-east-1.compute.amazonaws.com/emes/api/v2/workflow-result/exportResults?subProjectId="
RESULT_JSON = "http://ec2-54-46-0-164.ap-east-1.compute.amazonaws.com/emes/api/v2/workflow-result/exportOutputResults?subProjectId="
CONFIG_EXPORT = "http://ec2-54-46-0-164.ap-east-1.compute.amazonaws.com/emes/api/v2/getSubprojectMetaData?subProjectId="


subproject_list = [
    438, 439, 440, 441, 442, 443,
]

merge_dir = os.path.join(root_dir, 'merge')

In [ ]:
# def extract_date_from_project_name(project_name):
#     pattern = r"^2025(\d{4})_SIT$"
#     match = re.match(pattern, project_name)
    
#     if match:
#         date = match.group(1)  # 提取 xxxx
#         return date
#     else:
#         return 0000

def extract_date_from_project_name(project_name):
    pattern = r"^2025(\d{4})_UAT_Demo$"
    match = re.match(pattern, project_name)
    
    if match:
        date = match.group(1)  # 提取 xxxx
        return date
    else:
        return 0000

In [ ]:
import concurrent.futures
from concurrent.futures import ThreadPoolExecutor

# 多线程下载函数
def download_zip_files_mp(subproject_list, root_dir, overwirte=False, max_workers=4):
    zip_download_dict = {}

    def download_single(subproject_id):
        # 每个线程创建独立的session
        session = requests.Session()
        session.headers.update({
            "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64)"
        })

        download_url = RESULT_ALL+str(subproject_id)
        config_url = CONFIG_EXPORT+str(subproject_id)

        try:
            print(f"🔗 正在连接: {config_url}")
            response = session.get(config_url)
            response.raise_for_status()
            data = response.json()
            project_name = data['project']['name']
            date = extract_date_from_project_name(project_name)

            save_dir = os.path.join(root_dir, 'task'+date, 'results')
            os.makedirs(save_dir, exist_ok=True)

            print(f"🔗 正在连接: {download_url}")
            response = session.get(download_url, stream=True)
            response.raise_for_status()

            total_size = int(response.headers.get("content-length", 0))
            chunk_size = 1024 * 1024  # 1MB

            print(f'total size: {total_size/chunk_size} MB')

            parsed_url = urlparse(download_url)
            query_params = parse_qs(parsed_url.query)
            sub_project_id = query_params.get('subProjectId', ['unknown'])[0]
            filename = f"{sub_project_id}.zip"
            save_path = os.path.join(save_dir, filename)

            if not os.path.exists(save_path) or overwirte:
                with open(save_path, 'wb') as f, tqdm(
                    total=total_size,
                    unit='B',
                    unit_scale=True,
                    unit_divisor=1024,
                    desc=f"📥 下载 {filename} -> {save_dir}",
                    leave=True,
                ) as pbar:
                    for chunk in response.iter_content(chunk_size=chunk_size):
                        if chunk:
                            f.write(chunk)
                            pbar.update(len(chunk))

                print(f"✅ 下载完成: {save_path}\n")
                return subproject_id, save_path
            else:
                print(f"⚠️ 文件已存在，跳过下载: {save_path}\n")
                return subproject_id, save_path
        except Exception as e:
            print(f"❌ 下载失败: {download_url}\n原因: {e}")
            return subproject_id, None

    # 使用线程池执行下载任务
    with ThreadPoolExecutor(max_workers=max_workers) as executor:
        # 提交所有任务
        futures = {executor.submit(download_single, sid): sid for sid in subproject_list}

        # 收集结果
        for future in concurrent.futures.as_completed(futures):
            subproject_id, save_path = future.result()
            if save_path:
                zip_download_dict[subproject_id] = save_path

    return zip_download_dict

In [ ]:
# zip_download_dict = download_zip_files(subproject_list, root_dir)
zip_download_dict = download_zip_files_mp(subproject_list, root_dir)

In [ ]:
def simple_unzip(zip_path, dst_dir):
    print(f'{zip_path} unzip...')
    with zipfile.ZipFile(zip_path, 'r') as zip_ref:
        zip_ref.extractall(dst_dir)
    print(f'{zip_path} done\n')

In [ ]:
for key, zip_path in zip_download_dict.items():
    simple_unzip(zip_path, dst_dir=zip_path.replace('.zip', ''))


In [ ]:
for key, zip_path in zip_download_dict.items():
    esimage_merge(zip_path.replace('.zip', ''))

In [ ]:
for key, zip_path in zip_download_dict.items():
    yolo_dir = os.path.join(zip_path.replace('.zip', ''), 'yolo_dataset')
    img_dir = os.path.join(yolo_dir, 'images')
    
    print(f'{img_dir} selecting...')
    image_dir_select = img_dir+'_select'
    if os.path.exists(image_dir_select):
        continue
    shutil.rmtree(image_dir_select) if os.path.exists(image_dir_select) else None
    select_img(img_dir, image_dir_select, gap=10)
    if len(os.listdir(image_dir_select)) == 0:
        continue

    print(f'{image_dir_select} filtering...')
    image_dir_filter = img_dir+'_filter'
    shutil.rmtree(image_dir_filter) if os.path.exists(image_dir_filter) else None
    filter_deduplication(image_dir_select, image_dir_filter)

    print(f'process {img_dir}, {len(os.listdir(img_dir))} -> {len(os.listdir(image_dir_select))} -> {len(os.listdir(image_dir_filter))}')

In [ ]:
for key, zip_path in zip_download_dict.items():
    esresult2yolo(zip_path.replace('.zip', ''))

In [ ]:
for key, zip_path in zip_download_dict.items():
    yolo_dir = os.path.join(zip_path.replace('.zip', ''), 'yolo_dataset')
    img_dir = os.path.join(yolo_dir, 'images')

    select_defect(yolo_dir, yolo_dir+'_defect', anno_dir)

In [ ]:

merge_img_dir = os.path.join(merge_dir, 'images')
merge_label_dir = os.path.join(merge_dir, 'labels')

os.makedirs(merge_img_dir, exist_ok=True)
os.makedirs(merge_label_dir, exist_ok=True)

for key, zip_path in zip_download_dict.items():
    yolo_dir = os.path.join(zip_path.replace('.zip', ''), 'yolo_dataset_defect')
    img_dir = os.path.join(yolo_dir, 'images')
    label_dir = os.path.join(yolo_dir, 'labels')

    # 合并 images 目录
    print(f'processing {img_dir}')
    shutil.copytree(img_dir, merge_img_dir, dirs_exist_ok=True)
    # 合并 labels 目录
    print(f'processing {label_dir}')
    shutil.copytree(label_dir, merge_label_dir, dirs_exist_ok=True)
    


In [ ]:
def zip_folder_to_path(source_folder, destination_zip):
    with zipfile.ZipFile(destination_zip, 'w', zipfile.ZIP_DEFLATED) as zipf:
        for root, dirs, files in os.walk(source_folder):
            for file in files:
                file_path = os.path.join(root, file)
                # 在zip文件中创建相对路径
                arcname = os.path.relpath(file_path, start=source_folder)
                zipf.write(file_path, arcname)
    
    print(f"zip '{source_folder}' to '{destination_zip}'")

In [ ]:
zip_folder_to_path(merge_img_dir, merge_dir+'_image.zip')
zip_folder_to_path(merge_label_dir, merge_dir+'_label.zip')
